In [33]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import os
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import MinMaxScaler
import joblib
import json

In [2]:
data = pd.read_parquet("../data/features/feature_matrix.parquet")

# 1 IF


In [ ]:
model = IsolationForest(
    contamination=0.01, # Ожидаемая доля аномалий — 1%
    n_estimators=200, # Количество деревьев
    max_samples="auto", # Размер подвыборки для каждого дерева
    random_state=42,
    n_jobs=-1 # Используй все ядра
)

In [4]:
model.fit(data)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",200
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.01
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [5]:
scores = model.decision_function(data)
labels = model.predict(data)
data['anomaly_score'] = scores
data['anomaly_label'] = labels

In [6]:
n_anomalies = (data["anomaly_label"] == -1).sum()
print(f"Найдено аномалий: {n_anomalies} ({n_anomalies/len(data)*100:.2f}%)")

Найдено аномалий: 3681 (1.00%)


In [7]:
data_full = pd.read_parquet("../data/features/data_with_features.parquet")
data_full['anomaly_score'] = scores
data_full['anomaly_label'] = labels

In [11]:
top_20 = data_full.sort_values("anomaly_score", ascending=True).head(20)
print(top_20)

                    period                                        Регистратор  \
38540  2024-03-31 00:00:00         Операция 00БП-000006 от 31.03.2024 0:00:00   
38539  2024-03-31 00:00:00         Операция 00БП-000006 от 31.03.2024 0:00:00   
38588  2024-03-31 23:59:59  Регламентная операция 00БП-000018 от 31.03.202...   
84809  2024-06-30 23:59:59  Регламентная операция 00БП-000034 от 30.06.202...   
38541  2024-03-31 00:00:00         Операция 00БП-000006 от 31.03.2024 0:00:00   
178293 2024-12-31 23:59:59  Поступление (акт, накладная, УПД) 00БП-000084 ...   
38568  2024-03-31 23:59:59  Регламентная операция 00БП-000020 от 31.03.202...   
144086 2024-10-19 23:59:59        Операция 00БП-000020 от 19.10.2024 23:59:59   
38616  2024-03-31 23:59:59  Поступление (акт, накладная, УПД) 00БП-000013 ...   
366175 2025-12-28 00:00:00     Операции ЕНС 00БП-000001 от 28.12.2025 0:00:00   
144085 2024-10-19 23:59:59        Операция 00БП-000020 от 19.10.2024 23:59:59   
84844  2024-06-30 23:59:59  

# 2 LOF

In [ ]:
lof = LocalOutlierFactor(
    n_neighbors=50,
    contamination=0.01,
    novelty=False
)
lof_labels = lof.fit_predict(data)
lof_scores = lof.negative_outlier_factor_

/Users/rodion/VsCodeProjects/auditor/auditor/venv/lib/python3.13/site-packages/sklearn/neighbors/_lof.py:325: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(


In [15]:
data_full['lof_score'] = lof_scores
data_full['lof_label'] = lof_labels

# 3 ансамбль и скалер

скоры разные по шкале. IF даёт скоры от -0.14 до +0.25, LOF даёт от -20 до -1. Нельзя просто сложить.
Решение: привести оба к шкале [0, 1] где 1 = максимально аномально, потом усреднить.

In [18]:
# Шаг 1 — инвертируем IF скор (у него чем ниже = аномальнее)
if_scores = -data_full["anomaly_score"].values.reshape(-1, 1)

# Шаг 2 — инвертируем LOF скор (у него тоже чем ниже = аномальнее)
lof_scores_inv = -lof_scores.reshape(-1, 1)

In [21]:
scaler = MinMaxScaler()

iforest_normalized = scaler.fit_transform(-data["anomaly_score"].values.reshape(-1, 1))
lof_normalized = scaler.fit_transform(-lof_scores.reshape(-1, 1))
# Среднее
data_full["ensemble_score"] = (iforest_normalized.flatten() + lof_normalized.flatten()) / 2

In [26]:
top40 = data_full.nlargest(40, "ensemble_score")[
    ["period", "contractor", "account_dt", "account_kt", "amount", "ТипДокумента", "anomaly_score", "ensemble_score"]
]
print(top40)

                    period                                         contractor  \
157891 2024-11-18 16:00:00                             НОВГОРОДСКИЙ БЕКОН ООО   
157892 2024-11-18 16:00:00                             НОВГОРОДСКИЙ БЕКОН ООО   
157921 2024-11-18 16:00:00                             НОВГОРОДСКИЙ БЕКОН ООО   
158216 2024-11-18 16:17:09                       Путрашевич Роман Геннадьевич   
158215 2024-11-18 16:16:48                       Путрашевич Роман Геннадьевич   
38540  2024-03-31 00:00:00                                Внутренняя операция   
38539  2024-03-31 00:00:00                                Внутренняя операция   
38588  2024-03-31 23:59:59                                Внутренняя операция   
84809  2024-06-30 23:59:59                                Внутренняя операция   
38541  2024-03-31 00:00:00                                Внутренняя операция   
178293 2024-12-31 23:59:59                                         АЛБАСС ООО   
38568  2024-03-31 23:59:59  

In [31]:
data.columns

Index(['hour', 'day_of_week', 'month', 'is_weekend', 'is_night',
       'is_mounth_end', 'is_quarter_end', 'log_amount', 'is_negative_amount',
       'amount_zscore', 'is_amount_outlier', 'log_pair_frequency',
       'is_rare_pair', 'account_dt_freq', 'account_kt_freq',
       'account_pair_freq', 'log_contractor_frequency', 'is_first_operation',
       'is_first_contractor_pair', 'large_and_rare', 'late_and_large',
       'new_cont_and_large', 'manual_and_large', 'daily_volume',
       'hourly_volume', 'ТипДокумента_freq', 'is_manual'],
      dtype='object')

In [28]:
data_full.columns

Index(['period', 'Регистратор', 'contractor', 'КонтрагентИНН', 'account_dt',
       'СубконтоДт1', 'ВидСубконтоДт1', 'СубконтоДт2', 'ВидСубконтоДт2',
       'СубконтоДт3', 'ВидСубконтоДт3', 'account_kt', 'СубконтоКт1',
       'ВидСубконтоКт1', 'СубконтоКт2', 'ВидСубконтоКт2', 'СубконтоКт3',
       'ВидСубконтоКт3', 'dept_dt', 'dept_kt', 'amount', 'Содержание',
       'ТипДокумента', 'is_manual', 'hour', 'day_of_week', 'month',
       'is_weekend', 'is_night', 'is_mounth_end', 'is_quarter_end',
       'log_amount', 'is_negative_amount', 'abs_amount', 'account_pair',
       'pair_frequency', 'log_pair_frequency', 'is_rare_pair', 'pair_mean',
       'pair_std', 'amount_zscore', 'is_amount_outlier', 'contractor_freq',
       'log_contractor_frequency', 'contractor_ops_before',
       'is_first_operation', 'dept_dt_freq', 'dept_kt_freq', 'large_and_rare',
       'late_and_large', 'new_cont_and_large', 'manual_and_large',
       'first_contractor_pair', 'is_first_contractor_pair', 'date',
  

In [37]:
joblib.dump(model, "../models/isolation_forest_v1.pkl")
joblib.dump(lof, "../models/lof_v1.pkl")
joblib.dump(scaler, "../models/feature_scaler_v1.pkl")
# Сохрани также конфигурацию: список фичей, параметры модели, пороги
config = {
    "feature_cols": data.columns.tolist(),
    "contamination": 0.01,
    "model_type": "IsolationForest",
    "n_estimators": 200,
    "training_date": str(pd.Timestamp.now()),
    "training_rows": len(data),
}
with open("../models/config_v1.json", "w") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)